# Sample Creation

## Setup
The first part of this notebook defines all jubilee labware 
 -out of the 6 deck beds, 5 are in use: trashcan, tiprack, stock solutions, samples plate and the    aliquoted samples. Each one also has all of the necesary manual offsets for calibration puposes

In [ ]:
# this step reloads machine in order to prevent any error
%load_ext autoreload
%autoreload 2

In [3]:
# import statements
import requests

# ----------- Science Jubilee -------------
from science_jubilee import Machine as Jub
from science_jubilee.tools import Pipette
import time

import numpy as np
import pandas as pd
import os

In [ ]:
# defines the jubilee to call
jubilee = Jub.Machine(address='192.168.1.2', simulated = False) 

In [ ]:
# load deck
deck = jubilee.load_deck('lab_automation_deck.json')

In [ ]:
# this loads the well plate for the destination for samples (sample_holder_name, position)
samples = jubilee.load_labware('septavialrev1_44_holder_2000ul.json', 2)
# this line of code defines the boundary conditions of the well plate requires 3 points
upper_left = (19.9,183.2)
upper_right = (132.4, 183.4)
lower_right = (132.5, 113.0)
samples.manual_offset([upper_left,upper_right,lower_right])
samples.load_manualOffset()

In [ ]:
# this is same as lines above just for stock solution
stocks = jubilee.load_labware('20mlscintillation_12_wellplate_18000ul.json', 3)
upper_left = (167.4,176.8)
upper_right = (258.0, 176.8)
lower_right = (258.0, 119.8)
stocks.manual_offset([upper_left,upper_right,lower_right])
stocks.load_manualOffset()

In [ ]:
# same as above just for pipette tip rack
tiprack = jubilee.load_labware('opentrons_96_tiprack_300ul.json', 0)
upper_left = (27.0,82.7)
upper_right = (125.6,82.7)
lower_right = (125.8,19.4)
tiprack.manual_offset([upper_left,upper_right,lower_right])
tiprack.load_manualOffset()

In [ ]:
# defines position of trash can for tips
trash = jubilee.load_labware('agilent_1_reservoir_290ml.json', 1)

In [ ]:
deck.safe_z

In [ ]:
# this code block defines all of our stocks
water_stock = stocks[0].bottom(+5) 
blue1_stock = stocks[1].bottom(+5) #stock A: 0.10 M
red1_stock = stocks[2].bottom(+5)  #stock B: 0.10 M

# include however many we need, remember to include however many stocks there are

## Load Tools

In [ ]:
P300 = Pipette.Pipette.from_config(3,'Pipette','P300_config.json')
jubilee.load_tool(P300)

P300.add_tiprack(tiprack)
P300.trash = trash[0]

## Experiment Functions
 4 Main functions are written: transfer stock, transfer mix, transfer water and mixer
     transfer stock simply moves some volume from some stock to the final wellplate
     transfer mix includes the aliquot and mixing step that occurs at the final step of experimentprocess
     transfer water is just for water
     mixer just mixes and aliquots incase only that function is needed

In [ ]:
def transfer_stock(v, stock, target_sample):
    '''
    This function will take a volume to add, and from which stock to use

    Input - v (volume)(float)
          - stock (int) -position of the desired stock
          - sample (int) -position of the current sample
    Output - void, purpose of function    
    '''
    P300.transfer(v, source_well=stocks[stock].bottom(+3) ,
                  destination_well = samples[target_sample].bottom(+9),
                  blowout=True, 
                  new_tip='once' )

In [ ]:
def transfer_mix(v, stock, sample):
    '''
    This function will take a volume to add, and from which stock to use

    Input - v (volume)(float)
          - stock (int) index location of stock
          - sample (int) sample index location
 
    Output - void, purpose of function    
    '''
    # get the required material
    P300.transfer(v, source_well=stocks[stock].bottom(+3) ,
                  destination_well = samples[sample].bottom(+6),
                  blowout=True,
                  new_tip='always',
                  mix_after = (4,300) )

In [ ]:
def transfer_water(v,sample):
    '''
    This function will take a volume to add, and from which stock to use

    Input - v (volume)(float)
          - sample (int)
    Output - void, purpose of function    
    '''
    # source well is always the same
    P300.transfer(v, source_well=stocks[0].bottom(+3),
                  destination_well = samples[sample].bottom(+9),
                  blowout=True, 
                  new_tip='once' )

# Data for experiment #
the 2 main loops at the end account for the blocks that run the program above, the rest is all preparational information for the proper values to be input

## Run Experiment Code Lines

In [ ]:
df = pd.read_csv()
house_array = df.to_numpy()

In [ ]:
jubilee.pickup_tool(P300) # line to pickup tool

In [7]:
def cobalt(n_samples, k):
    row = 0
    for i in range(n_samples):
        transfer_stock(house_array[row, k],1 , i)
        row += 3

SyntaxError: invalid syntax (1199354809.py, line 4)

In [ ]:
def water(n_samples, k):
    row = 2
    for i in range(n_samples):
        transfer_water(house_array[row, k], i)
        row += 3

In [ ]:
def phoshpate(n_samples, k):
    row = 1
    for i in range(n_samples):
        transfer_mix(house_array[row, k], 2, i)
        row += 3

In [ ]:
n_samples = 6 # how hamy samples are being prepared
n_additions = 10 #how many Time Step Additions are being programmed
f1 = open('5_7_Jubilee_Timelog.txt', 'w' )
t_begin = time.time()
for k in range(n_additions):
    t_start = time.time()
    f1.write(time.ctime() + ": Start of Time Step" + str(k) +  "\n")
    cobalt(n_samples, k)
    f1.write(time.ctime() + ": End of Cobalt Addition; Time Elapsed: " + str(round(time.time()-t_start)) + "seconds\n")
    t_1 = time.time()
    water(n_samples, k)
    f1.write(time.ctime() + ': End of Water Addition; Time Elapsed: ' + str(round(time.time()-t_1)) + "seconds\n")
    t_2 = time.time()
    phoshpate(n_samples, k)
    t_end = time.time()
    f1.write(time.ctime() + ': End of Phosphate Mixing; Time Elapsed: ' + str(round(time.time()-t_2)) + "seconds. Time Taken for Step: " + str(round(time.time()-t_start)) + "seconds\n")

total_time = time.time() - t_begin
f1.write(time.ctime() + ": Total Time Elapsed:" + str(round(total_time)) + "seconds\n")
f1.close()

In [16]:
import time
f2 = open('5_8_Jubilee_Timelog.txt', 'w' )
t_start = time.time()
f2.write(time.ctime() + ': Start of Experiment\n')
time.sleep(2)
time_taken = time.time() - t_start
f2.write(time.ctime() + ': End of Loop. Time Elapsed:' + str(round(time_taken)) + ' seconds\n')
f2.close()
